In [ ]:
#| default_exp save_sighting


# Step 6: Save sightings to a database

![Step 6 diagram](https://raw.githubusercontent.com/jaewilson07/bird-watcher/main/docs/diagrams/06-step.png)

**Goal:** every bird we detect and identify gets logged to a SQLite database.

SQLite is built into Python — no server, just one file on disk. Perfect for a hobby project.

Two functions live in `bird_watcher/save_sighting.py`:

- `save_sighting_to_db` — insert one new sighting
- `list_sightings_from_db` — read recent sightings, optionally filtered by time

## Step 6.0 — Setup

In [ ]:
from pathlib import Path

import yaml

ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (root for root in ROOT_CANDIDATES if (root / "tutorials").exists()),
    Path.cwd(),
)
CONFIG_FILE = PROJECT_ROOT / "config.yaml"

CONFIG = {}
if CONFIG_FILE.exists():
    CONFIG = yaml.safe_load(CONFIG_FILE.read_text()) or {}
    if not isinstance(CONFIG, dict):
        raise TypeError(f"{CONFIG_FILE} must contain a top-level mapping")
else:
    print(f"Config file not found at {CONFIG_FILE}; using defaults.")

# Folder + file constants. _FOLDER = a directory, _FILE = a single file.
TUTORIALS_FOLDER = PROJECT_ROOT / "tutorials"
DATA_FOLDER = PROJECT_ROOT / "data"
SNAPSHOT_FOLDER = DATA_FOLDER / "snapshots"
CROP_FOLDER = DATA_FOLDER / "crops"
DB_FILE = DATA_FOLDER / "birds.db"
SAMPLE_BIRD_FILE = DATA_FOLDER / "samples" / "sample-bird.jpg"
MODEL_FILE = PROJECT_ROOT / "yolov8n.pt"

# From config.yaml (or hardcoded fallback)
PHONE_IP = str(CONFIG.get("phone_ip", "192.168.1.42"))
PHONE_URL = f"http://{PHONE_IP}:8080/photo.jpg"
SLACK_WEBHOOK = str(CONFIG.get("slack_webhook", ""))
HUGGINGFACE_API_KEY = str(CONFIG.get("huggingface_api_key", ""))

SNAPSHOT_FOLDER.mkdir(parents=True, exist_ok=True)
CROP_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Snapshot folder: {SNAPSHOT_FOLDER}")
print(f"Config file: {CONFIG_FILE}")
print(f"Phone URL: {PHONE_URL}")


## Step 6.1 — A table for sightings

Each row is one sighting: when, what species, how sure we were, which photo file.

In [ ]:
import sqlite3

DB_FILE.parent.mkdir(parents=True, exist_ok=True)
with sqlite3.connect(DB_FILE) as conn:
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS sightings (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp       TEXT    NOT NULL,
            species         TEXT    NOT NULL,
            confidence      REAL    NOT NULL,
            snapshot_file   TEXT    NOT NULL,
            bbox_x_min      INTEGER,
            bbox_y_min      INTEGER,
            bbox_x_max      INTEGER,
            bbox_y_max      INTEGER
        )
        """
    )
    conn.commit()
print(f"Table ready at {DB_FILE}")

## Step 6.2 — Insert a sighting (parameterized, never f-string)

Always use `?` placeholders for user-supplied data. F-strings in SQL = SQL injection vulnerabilities.

In [ ]:
from datetime import datetime

timestamp = datetime.now().isoformat(timespec="seconds")
with sqlite3.connect(DB_FILE) as conn:
    cursor = conn.execute(
        """
        INSERT INTO sightings (timestamp, species, confidence, snapshot_file)
        VALUES (?, ?, ?, ?)
        """,
        (timestamp, "Northern Cardinal", 0.92, "data/snapshots/2026-07-07_18-30-00.jpg"),
    )
    conn.commit()
    print("Inserted row id:", cursor.lastrowid)

## Step 6.3 — Wrap as `save_sighting_to_db`

In [ ]:
#| export
import sqlite3
from datetime import datetime
from pathlib import Path

def save_sighting_to_db(
    db_file: Path,
    snapshot_file: Path,
    species: str,
    confidence: float,
    bounding_box: dict | None = None,
    verbose: bool = True,
) -> int:
    """Add one new sighting to the database.

    Args:
        db_file: path to the SQLite file.
        snapshot_file: path to the photo file this sighting came from.
        species: the bird's name.
        confidence: how sure the classifier was, 0.0 - 1.0.
        bounding_box: optional dict with x_min, y_min, x_max, y_max.
        verbose: if True, print the new row id. Default True.

    Returns:
        The new row's id (auto-incremented).
    """
    import sqlite3
    from datetime import datetime

    db_file.parent.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().isoformat(timespec="seconds")
    with sqlite3.connect(db_file) as conn:
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS sightings (
                id              INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp       TEXT    NOT NULL,
                species         TEXT    NOT NULL,
                confidence      REAL    NOT NULL,
                snapshot_file   TEXT    NOT NULL,
                bbox_x_min      INTEGER,
                bbox_y_min      INTEGER,
                bbox_x_max      INTEGER,
                bbox_y_max      INTEGER
            )
            """
        )
        cursor = conn.execute(
            """
            INSERT INTO sightings (
                timestamp, species, confidence, snapshot_file,
                bbox_x_min, bbox_y_min, bbox_x_max, bbox_y_max
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                timestamp,
                species,
                confidence,
                str(snapshot_file),
                bounding_box.get("x_min") if bounding_box else None,
                bounding_box.get("y_min") if bounding_box else None,
                bounding_box.get("x_max") if bounding_box else None,
                bounding_box.get("y_max") if bounding_box else None,
            ),
        )
        conn.commit()
        row_id = cursor.lastrowid
    if verbose:
        print(f"Saved sighting #{row_id}: {species} ({confidence:.2f})")
    return row_id


## Step 6.4 — Add `list_sightings_from_db`

In [ ]:
#| export
import sqlite3
from pathlib import Path

def list_sightings_from_db(
    db_file: Path,
    since: str | None = None,
    limit: int = 100,
    verbose: bool = True,
) -> list[dict]:
    """Get sightings from the database, most recent first.

    Args:
        db_file: path to the SQLite file.
        since: optional ISO timestamp; only return sightings after this.
        limit: max rows to return.
        verbose: if True, print a count. Default True.

    Returns:
        List of sighting dicts, sorted by timestamp descending.
    """
    import sqlite3

    if not db_file.exists():
        return []
    with sqlite3.connect(db_file) as conn:
        conn.row_factory = sqlite3.Row
        if since:
            cursor = conn.execute(
                """
                SELECT * FROM sightings
                WHERE timestamp >= ?
                ORDER BY timestamp DESC
                LIMIT ?
                """,
                (since, limit),
            )
        else:
            cursor = conn.execute(
                "SELECT * FROM sightings ORDER BY timestamp DESC LIMIT ?",
                (limit,),
            )
        rows = [dict(row) for row in cursor.fetchall()]
    if verbose:
        print(f"{len(rows)} sighting(s) in {db_file.name}")
    return rows


## Acceptance criterion

In [ ]:
from bird_watcher.save_sighting import save_sighting_to_db, list_sightings_from_db

jpg_files = sorted(SNAPSHOT_FOLDER.glob("*.jpg"))
snapshot_file = SNAPSHOT_FOLDER / jpg_files[0].name if jpg_files else SNAPSHOT_FOLDER / "missing.jpg"

row_id = save_sighting_to_db(DB_FILE, snapshot_file, "Northern Cardinal", 0.92)
rows = list_sightings_from_db(DB_FILE)
assert any(r["id"] == row_id for r in rows), "New row should be in the listing"
print(f"✅ {len(rows)} sighting(s), most recent id={rows[0]['id']}")

## What's next

**Step 7:** open [07-slack.ipynb](07-slack.ipynb) — we'll send a Slack message whenever we save a sighting.